# 02 - Laser-driven nucleation

Two complementary lessons here:

1. **Topological protection.** A deterministic Gaussian field pulse cannot inject Hopf charge: continuous LLG dynamics preserve Q_H exactly, and a smooth field pulse is still continuous. We demonstrate this and show that with the same stability parameters from notebook 01, the system perturbs but returns to Q_H = 0.

2. **Stochastic nucleation.** Real femtosecond-laser experiments work by driving the electron bath to high effective temperatures, producing a chaotic spin state from which topological textures crystallize as the system cools. We simulate this here with a white-noise field burst (a stand-in for the two-temperature model coming in Phase B), and show Q_H = 1 hopfions nucleate from the cooling stage.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from hopfion.grid import Grid
from hopfion.field import uniform
from hopfion.topology import hopf_index
from hopfion.energy import EnergyParams, total_energy
from hopfion.laser import GaussianPulse
from hopfion.llg import LLGParams, integrate, llg_step_heun, relax
from hopfion.viz import slice_quiver

In [ ]:
g = Grid(48, 48, 48, 0.3, 0.3, 0.3, 'periodic')
ep = EnergyParams(A_ex=1.0, D=1.5, Ku=0.7, easy_axis=(0,0,1), H_ext=(0,0,0.0))
lp = LLGParams(gamma=1.0, alpha=0.1, dt=0.002)
m0 = uniform(g, direction=(0,0,1))
print(f'initial Q_H = {hopf_index(m0, g):+.3f}')

## Part 1: deterministic field pulse (will NOT nucleate)

Apply a strong ring-shaped field pulse and watch Q_H stay at zero throughout the dynamics.

In [ ]:
pulse = GaussianPulse(H0=(0,0,-15.0), t0=0.1, tau=0.04, profile='ring', width=0.6, ring_radius=1.5)
# Show pulse spatial envelope (z-slice)
env = np.asarray(pulse.spatial_envelope(g))
plt.imshow(env[..., g.nz//2].T, origin='lower', cmap='viridis',
           extent=[-g.nx*g.dx/2, g.nx*g.dx/2, -g.ny*g.dy/2, g.ny*g.dy/2])
plt.colorbar(label='envelope f(r)'); plt.title('Ring-shaped focal envelope of the pulse');

In [ ]:
Hf = pulse.field_factory(g)
m = uniform(g)
_, snaps, ts = integrate(m, g, ep, lp, n_steps=150, H_extra=Hf, snapshot_every=15)
Qs = [hopf_index(s, g) for s in snaps]
plt.plot(ts, Qs, 'o-'); plt.axhline(1, color='k', linestyle='--', alpha=0.3)
plt.xlabel('time'); plt.ylabel('Q_H'); plt.title('Q_H over time -- preserved by smooth deterministic field pulse'); plt.grid(True)

## Part 2: stochastic thermal burst (DOES nucleate)

Replace the deterministic pulse with a transient white-noise field representing electron-temperature spikes. This breaks topological protection at the discretization scale (real materials: through Bloch points) and lets hopfions form on cooling.

In [ ]:
def thermal_burst_then_relax(kT, seed, n_hot=60, n_cool=400):
    rng = np.random.default_rng(seed)
    sigma = (2.0 * lp.alpha * kT / (lp.dt * g.dV)) ** 0.5
    m = uniform(g)
    # 'hot' phase
    for _ in range(n_hot):
        H_n = np.asarray(rng.normal(size=(3, g.nx, g.ny, g.nz)) * sigma)
        m = llg_step_heun(m, g, ep, lp, H_extra=lambda x: H_n)
    Q_hot = hopf_index(m, g)
    # 'cool' phase
    m = relax(m, g, ep, n_steps=n_cool, dt=0.002)
    return m, Q_hot, hopf_index(m, g)

for kT in (5.0, 15.0, 30.0, 60.0):
    m_t, Q_hot, Q_cool = thermal_burst_then_relax(kT, seed=0)
    print(f'kT={kT:5.1f}: Q_hot={Q_hot:+.2f}, Q_cool={Q_cool:+.2f}, E_final={total_energy(m_t, g, ep):.1f}')

In [ ]:
# Visualize a nucleation success (kT=15 typically yields Q~1)
m_nuc, Q_hot, Q_cool = thermal_burst_then_relax(15.0, seed=0)
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
slice_quiver(m_nuc, g, plane='xy', stride=2, ax=axes[0]); axes[0].set_title(f'final, xy (Q_H={Q_cool:.2f})')
slice_quiver(m_nuc, g, plane='xz', stride=2, ax=axes[1]); axes[1].set_title('final, xz')